# 강의 03 · 실습 3 — RAG 검색기 구축 · (6) 고난도 III

## 1. 문제상황

- 구름월드 고객센터 검색기는 질문에 가까운 FAQ 청크(chunk)를 찾아 주지만, 손님에게는 청크 그대로가 아니라 답 문장이 나가야 합니다.
- 답에는 어느 FAQ 항목(행 번호와 카테고리)을 근거로 했는지 출처가 붙어야, 담당자가 나중에 확인할 수 있습니다.
- 시험 운영에서 임계값 1.5는 「표 물리면 얼마 돌려줘요」에 점수 1.47의 분실물 청크를 통과시켜 엉뚱한 근거가 붙었습니다. 담당자는 임계값을 1.3으로 낮추기로 했습니다.
- 그러면 「표 물리면 얼마 돌려줘요」처럼 표현이 달라 못 찾은 질문이 컷되므로, 컷된 질문을 바로 모른다고 하지 말고 질의를 한 번 바꿔 다시 검색한 뒤에도 근거가 없을 때만 모른다고 답하기를 원합니다.

## 2. 문제와 목표

- **문제**: 검색 결과가 청크로 끝나 답 문장과 출처가 없고, 컷된 질문을 표현 차이인지 문서 밖인지 가리지 않고 모두 모른다고 처리합니다.
- **목표**: 상위 청크를 프롬프트에 동봉해 모델이 근거로만 답 문장을 만들고 출처(행 번호·카테고리)를 병기하며, 컷된 질문은 모델에게 질의를 한 번 다시 쓰게 해 재검색하고 그래도 컷이면 고정 안내 문장으로 보내는 프로그램을 만듭니다.
    - FAQ 파일: `day05_faq_구름월드.csv`(32행). 저장 디렉터리: `chroma_db`.
    - 적재는 행마다 `[카테고리] Q: … A: …` 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터로 합니다. 출처는 이 메타데이터로 만듭니다.
    - 임계값: 1.3. 재검색: 한 번.
    - 답 생성과 질의 재작성에 쓰는 모델: `openai/gpt-5.6-luna`(litellm 경유).
    - 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」 출처 형식: `[행 N · 카테고리]`.
    - 질문 세 개: 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 문서 안 질문의 답에 FAQ 내용과 출처(행 번호와 카테고리)가 붙고, 표현이 다른 질문(「표 물리면 얼마 돌려줘요?」)은 재작성 후 통과해 답이 나오며, 문서 밖 질문은 재작성 후에도 컷되어 고정 안내 문장이 나오는 것을 출력에서 확인합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기와 `.env` 읽기는 주어진 것입니다. 모델 객체는 직접 만듭니다. 이 실습이 쓰는 모델과 그 모델을 만드는 코드 한 줄은 아래 셀 끝에 주석으로 적어 두었습니다. 아래 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.


In [2]:
import csv
import os

from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

# 모델 — 이 실습이 쓰는 모델
#   채팅 모델: llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
#   임베딩 모델: emb = OpenAIEmbeddings(model="text-embedding-3-small")


준비를 마쳤습니다.


### 단계 ① — 문서 적재 (요구사항 1)

- 검색 단위는 `Document`입니다. 본문(`page_content`)이 임베딩되어 검색에 쓰이고, 메타데이터(`metadata`)는 청크와 함께 돌아와 출처 표시나 필터에 쓰입니다.
- FAQ 한 행을 청크 하나로 삼습니다. 질문과 답을 한 본문에 넣어야 질문 표현으로 검색해도 답이 함께 돌아옵니다.

In [3]:
CSV_PATH = "day05_faq_구름월드.csv"

docs = []
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    for i, row in enumerate(csv.DictReader(f), start=1):
        if not (row.get("Question") or "").strip():
            continue
        text = f"[{row['Category']}] Q: {row['Question']}\nA: {row['Answer']}"
        docs.append(Document(page_content=text, metadata={"row": i, "category": row["Category"]}))

print("적재 문서 수:", len(docs))
print("첫 문서:", docs[0].page_content[:60], "| 메타데이터:", docs[0].metadata)

적재 문서 수: 32
첫 문서: [이용정보] Q: 구름월드 운영시간이 어떻게 되나요?
A: 구름월드는 매일 09:30~21:00에 운영합니다 | 메타데이터: {'row': 1, 'category': '이용정보'}


### 단계 ② — 임베딩 준비 (요구사항 2)

- 임베딩은 문장을 숫자 벡터로 바꾸는 모델입니다. 의미가 가까운 문장은 벡터도 가깝습니다.
- 문서를 넣을 때와 질문을 넣을 때 같은 임베딩을 써야 같은 좌표계에서 거리를 잴 수 있습니다.
- 답 생성용 모델은 임베딩과 별개의 부품입니다. 검색에는 쓰이지 않고 동봉된 근거로 답 문장을 만들 때만 쓰입니다.

In [7]:
emb = OpenAIEmbeddings(model="text-embedding-3-small")

vec = emb.embed_query("자유이용권 환불이 되나요?")
print("벡터 길이:", len(vec), "| 앞 세 값:", [round(v, 4) for v in vec[:3]])

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("답 생성 모델 준비를 마쳤습니다.")

벡터 길이: 1536 | 앞 세 값: [-0.0184, 0.0085, -0.0332]
답 생성 모델 준비를 마쳤습니다.


### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- `Chroma.from_documents`가 문서마다 임베딩을 계산해 저장소에 넣습니다. `persist_directory`를 주면 디렉터리에 남아 프로그램이 끝나도 유지됩니다.
- 문서 id를 행 번호로 주면 같은 셀을 다시 실행해도 같은 id에 덮어써 항목이 늘지 않습니다.

In [8]:
db = Chroma.from_documents(
    docs, emb,
    persist_directory="chroma_db",
    ids=[f"row-{d.metadata['row']}" for d in docs],
)

print("저장된 항목 수:", len(db.get()["ids"]))

저장된 항목 수: 32


### 단계 ④ — 유사도 검색과 답 생성 (요구사항 4)

- 검색은 FAQ 검색기와 같습니다. 답 생성은 청크 본문을 「근거」로 동봉한 프롬프트로 모델을 한 번 부르는 일입니다.
- 출처는 청크의 메타데이터에서 읽습니다. 행 번호와 카테고리를 답 끝에 붙입니다.

In [9]:
def retrieve(question: str, k: int = 3):
    """청크와 점수의 목록을 돌려준다."""
    return db.similarity_search_with_score(question, k=k)


def generate(question: str, hits) -> str:
    """상위 청크를 근거로 동봉해 답 문장을 만들고 1위 청크의 출처를 붙인다."""
    context = "\n---\n".join(d.page_content for d, _ in hits)
    res = llm.invoke(f"아래 근거로만 답하라. 근거에 없는 내용은 말하지 마라.\n근거:\n{context}\n질문: {question}")
    top = hits[0][0].metadata
    return f"{res.content.strip()} [행 {top['row']} · {top['category']}]"


hits = retrieve("자유이용권 환불 규정 알려 주세요")
print("1위 score:", round(hits[0][1], 4))
print(generate("자유이용권 환불 규정 알려 주세요", hits))

1위 score: 0.7352
이용일 전날까지 취소하면 전액 환불되고, 이용일 당일 취소하면 50%만 환불됩니다. 이용 개시 후에는 환불되지 않습니다. [행 8 · 티켓]


### 단계 ⑤ — 임계값 컷과 처분 두 경로 (요구사항 5)

- 첫 컷은 표현 차이일 수 있으므로 모델에게 질의를 다시 쓰게 해 한 번 재검색합니다. 재검색도 컷이면 문서 밖 질문으로 보고 고정 안내 문장으로 보냅니다.
- 재시도 상한을 두어야 루프가 끝납니다. 여기서는 한 번입니다.

In [14]:
THRESHOLD = 1.3
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."


def answer(question: str) -> str:
    """통과면 답 생성, 컷이면 질의 재작성 후 한 번 재검색, 그래도 컷이면 고정 안내 문장."""
    query = question
    for attempt in (1, 2):
        hits = retrieve(query)
        s = hits[0][1]
        verdict = "통과" if s <= THRESHOLD else "컷"
        print(f"  [{attempt}회차] query={query!r} → 1위 score={s:.4f} {verdict}")
        if s <= THRESHOLD:
            return generate(question, hits)
        if attempt == 1:
            rq = llm.invoke(f"다음 질문을 시설 FAQ 검색에 더 잘 걸리도록 한 문장으로 바꿔라. 바꾼 질문만 출력하라.\n질문: {question}")
            query = rq.content.strip()
    return NO_EVIDENCE


for q in [
            # "자유이용권 환불 규정 알려 주세요",
            # "표 물리면 얼마 돌려줘요?",
            # "파이썬 리스트 정렬은 어떻게 하나요?",
            "샴쌍둥이는 두명요금인가요 한명요금인가요?",
             ]:
    print(f"[질문] {q}")
    print(f"  [답] {answer(q)}")
    print()

[질문] 샴쌍둥이는 두명요금인가요 한명요금인가요?
  [1회차] query='샴쌍둥이는 두명요금인가요 한명요금인가요?' → 1위 score=1.4461 컷
  [2회차] query='샴쌍둥이의 입장료는 1명 요금인가요, 2명 요금인가요?' → 1위 score=1.3205 컷
  [답] 문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.



## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 문서 안 질문은 첫 검색에서 통과하고, 답 끝에 행 번호와 카테고리 출처가 붙습니다.
2. 표현이 다른 질문은 첫 검색에서 컷되고, 다시 쓴 질의로 통과해 환불 규정 답과 출처가 나옵니다.
3. 문서 밖 질문은 다시 쓴 질의로도 컷되어 고정 안내 문장이 나옵니다.

세 가지가 모두 확인되면 완성입니다.